In [1]:
from google.colab import files

uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset (1).csv


In [3]:
import pandas as pd
import re


df = pd.read_csv("/content/IMDB Dataset.csv")
df


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [6]:
df.isnull().sum()

,0
review,0
sentiment,0


In [8]:
df.duplicated().sum()

np.int64(418)

In [9]:
df.drop_duplicates()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [11]:
## text preprocessing

##1 --> converting to lowercase
df['review'] = df['review'].str.lower()



In [12]:
##2 --> removing urls
def remove_urls(text):
    text =re.sub(r"http\S+", "", text)
    return text

df['review'] = df['review'].apply(remove_urls)


In [13]:
##3 --> removing punctuation and additional symbols

def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]" , "" , text) ## A-Z , a-z , \s
    return text

df['review'] = df['review'].apply(remove_punctuations)

In [14]:
##4 --> html pattern
def remove_html(text):
    text = re.sub(r"<.*?>" , "" , text)
    return text

df['review'] = df['review'].apply(remove_html)

In [15]:
%pip install nltk

In [16]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [17]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [18]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text =text.replace(word, "")

    return text

df['review'] = df['review'].apply(remove_stopwords)

In [19]:
from nltk.stem import PorterStemmer

In [20]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " " .join(stemmed_words)
df['review'] = df['review'].apply(stemming)

In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['sentiment'] = le.fit_transform(df["sentiment"])

In [25]:
y = df["sentiment"]

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])


In [23]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4088388 stored elements and shape (50000, 5000)>

In [26]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
        X ,y , test_size =0.2 , random_state = 42
)

In [27]:
import torch
from torch.utils.data import DataLoader , TensorDataset

In [30]:
train_set = TensorDataset(
    torch.tensor(X_train.toarray() , dtype = torch.float32),
    torch.tensor(y_train.values , dtype = torch.float32)
)

test_set = TensorDataset(
    torch.tensor(X_test.toarray() , dtype = torch.float32),
    torch.tensor(y_test.values , dtype = torch.float32)
)

In [31]:
train_loader = DataLoader(train_set, shuffle = True , batch_size =64)
test_loader = DataLoader(test_set , shuffle = True , batch_size = 64)

In [32]:
import torch
import torch.nn as nn
import torch.optim as optimizer

In [33]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        # Initial hidden state
        h0 = torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size
        )

        # RNN
        out, _ = self.rnn(x, h0)

        # Take output from the last timestamp
        out = self.fc(out[:, -1, :])

        return out

In [34]:
import torch.optim as optim

input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [35]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb ,yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)  ## add singelton direction

        outputs = model(Xb)

        outputs=torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs , yb)
        loss.backward()
        optimizer.step()

    print(f"{epoch}/{epochs} and loss = {loss.item()}")



0/10 and loss = 0.25995588302612305
1/10 and loss = 0.2534107267856598
2/10 and loss = 0.18953920900821686
3/10 and loss = 0.4190261960029602
4/10 and loss = 0.25979480147361755
5/10 and loss = 0.22701625525951385
6/10 and loss = 0.16805392503738403
7/10 and loss = 0.19850575923919678
8/10 and loss = 0.18465521931648254
9/10 and loss = 0.21641524136066437


In [37]:
model.eval()

with torch.no_grad():

    correct_vals =0
    total_val = 0

    for Xb , yb in test_loader:
        Xb =Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        total_val += yb.size(0)
        correct_vals += (predicted==yb).sum().item()

    print(f"accuracy={correct_vals/total_val*100}")

accuracy=85.72


In [38]:
torch.save(model.state_dict(), "sentiment_rnn.pth")

In [39]:
import joblib

joblib.dump(tf, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [40]:
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']